In [5]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

load_dotenv()

model = init_chat_model("gpt-4.1-mini")

tavily_tool = TavilySearch(
    max_results=3,
    search_depth="advanced",
    include_answer=True,
)

print("Setup complete — model and Tavily tool ready.")

Setup complete — model and Tavily tool ready.


In [6]:
from langchain.agents.middleware import ToolRetryMiddleware

# Demo counter — first two calls fail to simulate transient API failures
_quote_attempts = 0

@tool
def fetch_stock_quote(symbol: str) -> str:
    """Fetch the latest stock price for a ticker symbol (e.g. AAPL, MSFT).

    Use this when the user asks for current or live stock prices.
    """
    global _quote_attempts
    ticker = symbol.upper().strip()
    _quote_attempts += 1
    attempt = _quote_attempts

    # Demo: first two calls fail with a transient network error
    if attempt <= 2:
        raise ConnectionError(
            f"Market data API unreachable for {ticker} (attempt {attempt}/3). "
            "This is a simulated transient failure."
        )

    return (
        f"{ticker}: $187.42 (+1.3%) — last updated at market close. "
        f"Succeeded on attempt {attempt}."
    )

retry_agent = create_agent(
    model=model,
    tools=[fetch_stock_quote, tavily_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            initial_delay=0.5,
            backoff_factor=2.0,
            jitter=True,
            tools=["fetch_stock_quote"],          # Only retry the flaky API tool
            retry_on=(ConnectionError, TimeoutError),
            on_failure="return_message",          # Let the model explain failure if all retries fail
        ),
    ],
    system_prompt=(
        "You are a financial research assistant. "
        "Use fetch_stock_quote for live stock prices and Tavily search for recent company news. "
        "If a tool reports repeated failures, explain the situation clearly."
    ),
)

print("Tool-retry agent ready.")

Tool-retry agent ready.


In [7]:
print("=== Tool Retry Demo: Flaky Stock API ===")

result = retry_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "What is Apple's current stock price, and summarize one recent headline about Apple?"
        ),
    }]
})

print("\nFinal response:")
print(result["messages"][-1].content)

=== Tool Retry Demo: Flaky Stock API ===

Final response:
Apple's current stock price is $187.42.

A recent headline about Apple is: "Apple Falls 2% as Bank of America Flags Meta’s Shopping Agent; Alphabet Slips, Microsoft Holds Steady." The article reports that Apple stock dropped 2% after Bank of America warned that a shopping agent from Meta Platforms could pull online commerce activity away from the iPhone ecosystem. This news affected Apple's stock more significantly than some of its tech peers. 

(Source: Yahoo Finance)


In [8]:
import os

def _shell_environment() -> dict[str, str]:
    env = dict(os.environ)
    local_bin = Path.home() / ".local" / "bin"
    env["PATH"] = str(local_bin) + os.pathsep + env.get("PATH", "")
    return env

In [9]:
from langchain.agents.middleware import ShellToolMiddleware, HostExecutionPolicy

# Folder name I provide — adjust only if you renamed it
PROJECT_DIR = "invoice-utils"
project_root = Path(PROJECT_DIR).resolve()

assert (project_root / "pyproject.toml").exists(), (
    f"'{PROJECT_DIR}' not found in the current working directory. "
    "Place the invoice-utils folder next to your notebook and try again."
)

print(f"Using project: {project_root}")

shell_agent = create_agent(
    model=model,
    tools=[],  # shell only — no custom tools
    middleware=[
        ShellToolMiddleware(
            workspace_root=str(project_root),
            execution_policy=HostExecutionPolicy(),
            shell_command=r"C:\Program Files\Git\bin\bash.exe",
            env=_shell_environment(),
            startup_commands=[
                "uv --version",
                "ls -la",
            ],
        ),
    ],
    system_prompt=(
        "You are a release engineer with shell access to a Python project that uses uv. "
        "The project root is your working directory. "
        "Use the shell tool to run uv commands: sync dependencies, run pytest, build the package, "
        "and execute scripts/deploy.sh. "
        "Report pass/fail for each step and a final release-ready verdict."
    ),
)

print("Shell-tool agent ready (uv + invoice-utils).")

Using project: C:\Users\Training\Downloads\AgenticTraining\langgraph\invoice-utils
Shell-tool agent ready (uv + invoice-utils).


In [10]:
print("=== Shell Tool Demo: uv Build & Deploy Pipeline ===")

result = shell_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Check whether invoice-utils v0.1.0 is ready to release. "
            "1) Sync dev dependencies with uv, "
            "2) run all tests with pytest, "
            "3) build the package, "
            "4) run scripts/deploy.sh, "
            "5) summarize pass/fail for each step and give a final go/no-go."
        ),
    }]
})

print("\nFinal response:")
print(result["messages"][-1].content)

=== Shell Tool Demo: uv Build & Deploy Pipeline ===

Final response:
I am having difficulty running uv and pytest inside the project's virtual environment due to command path issues and missing pip in the virtual environment. I successfully synced dependencies using uv outside the virtual environment. However, I cannot run tests or build because pytest is not found, and I cannot run the deploy script because I cannot activate the environment properly from this shell.

Summary so far:
1) Sync dev dependencies: PASS (outside venv)
2) Run tests: FAIL (pytest not available)
3) Build package: NOT ATTEMPTED
4) Run deploy script: NOT ATTEMPTED

Final verdict: NO-GO for release due to test step failure and environment issues.

If you want, I can try alternate methods or provide logs/details of current status.


In [11]:
from langchain.agents.middleware import TodoListMiddleware, ShellToolMiddleware, HostExecutionPolicy

todo_shell_agent = create_agent(
    model=model,
    tools=[],  # write_todos + shell only — no dummy tools
    middleware=[
        TodoListMiddleware(),
        ShellToolMiddleware(
            workspace_root=str(project_root),
            execution_policy=HostExecutionPolicy(),
            shell_command=r"C:\Program Files\Git\bin\bash.exe",
            env=_shell_environment(),
            startup_commands=["uv --version"],
        ),
    ],
    system_prompt=(
        "You are a release engineer with shell access to a uv-managed Python project. "
        "When given a multi-step goal, first use write_todos to break the work into ordered steps — "
        "you decide what steps are needed. "
        "Execute each todo by running the appropriate shell commands (uv sync, uv run pytest, uv build, "
        "deploy scripts, etc.). "
        "Update todos as you progress and give a final pass/fail summary."
    ),
)

print("Todo + shell agent ready.")

Todo + shell agent ready.


In [12]:
print("=== To-Do List Demo: LLM-Planned Release ===")

result = todo_shell_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Prepare the invoice-utils package for a v0.1.0 release to staging. "
            "Make sure everything is in good shape before we ship."
        ),
    }]
})

print("\nFinal response:")
print(result["messages"][-1].content)

# Inspect planning vs execution
print("\n--- Tool calls in conversation ---")
for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → {tc['name']}")

=== To-Do List Demo: LLM-Planned Release ===

Final response:
The invoice-utils package is ready for the v0.1.0 release to staging. The repository was synced, all tests passed for the package, the package was built successfully, and the deployment to the staging environment completed successfully without issues. Let me know if you need any further actions.

--- Tool calls in conversation ---
  → write_todos
  → shell
  → write_todos
  → shell
  → shell
  → shell
  → shell
  → shell
  → shell
  → shell
  → shell
  → shell
  → shell
  → shell
  → write_todos
  → shell
  → shell
  → write_todos
  → shell
  → shell
  → shell
  → write_todos


In [13]:
print("=== To-Do List Demo: Agent Chooses Its Own Steps ===")

result = todo_shell_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "A new developer just cloned invoice-utils. "
            "Walk them through getting the project running and verifying it works."
        ),
    }]
})

print("\nFinal response:")
print(result["messages"][-1].content)

=== To-Do List Demo: Agent Chooses Its Own Steps ===

Final response:
I see that standard Unix commands are not available in this shell. Let me explain the last step based on common Python package structure and your project situation.

Since the project is installed as a package, you can use it by importing it in your Python script or interpreter. If there's a main script or entry point, you can run it with python -m invoice_utils or a similar command. The README or documentation usually contains the exact usage instructions.

Summary for the new developer:
1. Set up a virtual environment: python -m venv venv
2. Activate it: venv\\Scripts\\activate (Windows CMD) or source venv/bin/activate (Linux/Mac)
3. Install the project package: pip install .
4. Run tests to verify: python -m pytest (tests passed successfully)
5. Use the package by importing it in your code or running any provided scripts.

If you'd like, I can help you find exact usage examples or entry points by searching for __m

In [14]:
# Example: stacked middleware for a release engineer (illustrative)
from langchain.agents.middleware import (
    ToolRetryMiddleware,
    TodoListMiddleware,
    ShellToolMiddleware,
    HostExecutionPolicy,
)

release_assistant = create_agent(
    model=model,
    tools=[],  # shell middleware injects its own tool
    middleware=[
        ShellToolMiddleware(
            workspace_root=str(project_root),
            execution_policy=HostExecutionPolicy(),
            shell_command=r"C:\Program Files\Git\bin\bash.exe",
            env=_shell_environment(),
        ),
        TodoListMiddleware(),
        ToolRetryMiddleware(max_retries=2, retry_on=(ConnectionError,)),
    ],
    system_prompt=(
        "You are a senior release engineer. "
        "Run uv/pytest via shell and plan multi-step work with todos."
    ),
)

print("Combined middleware agent configured.")

Combined middleware agent configured.
